# Final Project: UK Solar Electricity Forecasting

This notebook contains the final project for the Time Series course, analyzing and forecasting UK solar electricity generation.

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ydata_profiling import ProfileReport
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 6)
sns.set_theme(style='whitegrid')

In [ ]:
# Load the dataset
df = pd.read_csv('../../UK_electricity_solar_2021_2024.csv', parse_dates=['settlement_date'], index_col='settlement_date')
df.head()

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Quick visualization
plt.figure(figsize=(14, 6))
plt.plot(df.index, df['embedded_solar_generation'])
plt.title('UK Solar Electricity Generation (2021-2024)')
plt.xlabel('Date')
plt.ylabel('Generation (MW)')
plt.show()

In [ ]:
# Generate a profiling report (can take a minute on large datasets)
# profile = ProfileReport(df, title='Solar Profiling Report', minimal=True)
# profile.to_notebook_iframe()

## 3. Time Series Analysis
Check for stationarity and decompose the series.

In [ ]:
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose

# Resample to daily frequency to smooth the half-hourly data and speed up modeling
df_daily = df.resample('D').mean()
df_daily.dropna(inplace=True)

# Seasonal Decomposition
result = seasonal_decompose(df_daily['embedded_solar_generation'], model='additive', period=365)
fig = result.plot()
fig.set_size_inches(12, 8)
plt.show()

In [ ]:
def adf_test(series):
    print('Augmented Dickey-Fuller Test:')
    result = adfuller(series.dropna())
    print(f'ADF Statistic: {result[0]:.4f}')
    print(f'p-value: {result[1]:.4f}')
    if result[1] <= 0.05:
        print('Reject the null hypothesis (H0), the data is stationary.')
    else:
        print('Fail to reject the null hypothesis (H0), the data is non-stationary.')

adf_test(df_daily['embedded_solar_generation'])

### Autocorrelation (ACF & PACF)

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

fig, axes = plt.subplots(1, 2, figsize=(16, 4))
plot_acf(df_daily['embedded_solar_generation'].dropna(), ax=axes[0], lags=40)
plot_pacf(df_daily['embedded_solar_generation'].dropna(), ax=axes[1], lags=40)
plt.show()

## 4. Feature Engineering
Create basic time-based features.

In [ ]:
df_daily['month'] = df_daily.index.month
df_daily['day_of_week'] = df_daily.index.dayofweek
# Add lag variable based on ACF
df_daily['lag_1'] = df_daily['embedded_solar_generation'].shift(1)
df_daily.dropna(inplace=True)
df_daily.head()

## 5. Modeling (ARIMA)
Split into train and test sets chronologically.

In [ ]:
train_size = int(len(df_daily) * 0.8)
train, test = df_daily.iloc[:train_size], df_daily.iloc[train_size:]

plt.figure(figsize=(12, 5))
plt.plot(train.index, train['embedded_solar_generation'], label='Train')
plt.plot(test.index, test['embedded_solar_generation'], label='Test')
plt.legend()
plt.title('Train/Test Split')
plt.show()

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
import statsmodels.api as sm

# Note: Pass the original (non-differenced) data. The model handles differencing internally via `d`.
model = ARIMA(train['embedded_solar_generation'], order=(7, 1, 1))
results = model.fit()
print(results.summary())

## 6. Evaluation & Forecasting

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

# Forecast on the test set length
forecast_obj = results.get_forecast(steps=len(test))
forecast_mean = forecast_obj.predicted_mean
conf_int = forecast_obj.conf_int()

# Metrics
mae = mean_absolute_error(test['embedded_solar_generation'], forecast_mean)
rmse = np.sqrt(mean_squared_error(test['embedded_solar_generation'], forecast_mean))
mape = mean_absolute_percentage_error(test['embedded_solar_generation'], forecast_mean)

print(f'MAE:  {mae:.2f}')
print(f'RMSE: {rmse:.2f}')
print(f'MAPE: {mape:.2%}')

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(train.index[-100:], train['embedded_solar_generation'][-100:], label='Train (last 100 days)')
plt.plot(test.index, test['embedded_solar_generation'], label='Test Actuals')
plt.plot(test.index, forecast_mean, label='Forecast', color='red')
plt.fill_between(test.index, conf_int.iloc[:, 0], conf_int.iloc[:, 1], color='red', alpha=0.2, label='Confidence Interval')
plt.title('ARIMA Forecast vs Actuals')
plt.legend()
plt.show()